In [17]:
!cp -r /kaggle/input/* /kaggle/working/rag_storage
!pip install streamlit llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface llama-index-vector-stores-faiss transformers torch accelerate faiss-cpu pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of huggingface-hub[inference] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of huggingface-hub[inference] to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-index-cli to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multipl

In [18]:
%%writefile app.py
import streamlit as st
import torch
import os
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from llama_index.core import Settings, StorageContext, load_index_from_storage
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore

st.set_page_config(page_title="Thai Food Recipe Assistant", page_icon="🍲")
st.title("🍲 ผู้ช่วยค้นหาสูตรอาหารไทย")

@st.cache_resource(show_spinner="กำลังโหลดโมเดล AI ลง GPU...")
def load_rag_system():
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-m3", 
        max_length=512
    )

    model_name = "iapp/chinda-qwen3-4b"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name, 
        torch_dtype="auto", 
        device_map="auto", 
        trust_remote_code=True
    )

    system_prompt = (
        "คุณคือผู้ช่วยค้นหาและแนะนำวิธีทำอาหารไทย ที่มีข้อมูลใน documents\n"
        "ตอบเป็นภาษาไทยเสมอ\n"
        "ตอบคำถามแค่ที่ถาม เช่น ถามวิธีทำตอบแค่วิธีทำ ถามวัตถุดิบตอบแค่วัตถุดิบ\n"
        "ขอคำตอบแค่ 3 หัวข้อ คือ วัตถุดิบ เครื่องปรุง วิธีทำ\n"
        "ใช้ข้อมูล วัตถุดิบ เครื่องปรุง วิธีทำ แค่ใน documents เท่านั้น\n"
        "ห้ามนำสูตรอื่นมาดัดแปลงหรืออนุมานเองเด็ดขาด (เช่น ถามผัดไทย ห้ามนำผัดไทยไชยามาตอบแทน)\n"
        "ห้ามตอบจากความรู้ของตัวเองโดยเด็ดขาด\n"
        "เมื่อตอบครบถ้วนแล้วให้จบคำตอบทันที ห้ามวนลูปพิมพ์ข้อมูลซ้ำเด็ดขาด\n"
        "ถ้าไม่มีชื่อเมนูที่ตรงกับคำถามเป๊ะๆ ใน documents ให้ตอบว่าไม่พบข้อมูล"
    )

    def custom_messages_to_prompt(messages):
        formatted_prompt = ""
        for i, msg in enumerate(messages):
            if i == 0:
                content = f"{system_prompt}\n\nคำถาม: {msg.content}"
                formatted_prompt += f"<|im_start|>user\n{content}<|im_end|>\n"
            else:
                role = "user" if msg.role == "user" else "assistant"
                formatted_prompt += f"<|im_start|>{role}\n{msg.content}<|im_end|>\n"
        formatted_prompt += "<|im_start|>assistant\n"
        return formatted_prompt

    def completion_to_prompt(prompt):
        return f"<|im_start|>user\n{system_prompt}\n\nคำถาม: {prompt}<|im_end|>\n<|im_start|>assistant\n"

    Settings.llm = HuggingFaceLLM(
        model=model,
        tokenizer=tokenizer,
        context_window=4096,
        max_new_tokens=512, 
        generate_kwargs={"temperature": 0.3, 
                         "top_p": 0.95, 
                         "do_sample": True,
                         "repetition_penalty": 1.12,
                         "eos_token_id": tokenizer.eos_token_id
                        },
        messages_to_prompt=custom_messages_to_prompt,
        completion_to_prompt=completion_to_prompt,
        is_chat_model=True,
    )

    vector_store = FaissVectorStore.from_persist_dir("./rag_storage")
    storage_context = StorageContext.from_defaults(vector_store=vector_store, persist_dir="./rag_storage")
    index = load_index_from_storage(storage_context=storage_context)

    return index.as_chat_engine(chat_mode="context", similarity_top_k=5)

chat_engine = load_rag_system()

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("พิมพ์คำถามที่นี่ เช่น ขอวิธีทำแกงเขียวหวาน"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("กำลังค้นหาสูตรจากฐานข้อมูล..."):
            response = chat_engine.chat(prompt)

            # ตัด <think> ออก (เพิ่ม |$ เผื่อกรณีแท็กปิดไม่สมบูรณ์)
            answer = re.sub(r'<think>.*?(</think>|$)', '', response.response, flags=re.DOTALL).strip()
            
            # ให้แสดงผลแค่ answer ที่กรองแล้ว (ลบ st.markdown ตัวบนออก)
            st.markdown(answer)

    # ตอนบันทึกประวัติแชท ก็ต้องบันทึกตัวที่กรองแล้วเท่านั้น
    st.session_state.messages.append({"role": "assistant", "content": answer})

Overwriting app.py


In [19]:
import os
import shutil

# ค้นหาโฟลเดอร์ที่มีไฟล์ฐานข้อมูลของคุณ
source_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'default__vector_store.json' in files or 'docstore.json' in files:
        source_dir = root
        break

if source_dir:
    print(f"✅ พบฐานข้อมูลต้นฉบับซ่อนอยู่ที่: {source_dir}")
    
    # ลบโฟลเดอร์ rag_storage เดิมที่อาจจะผิดพลาดทิ้งไปก่อน
    if os.path.exists('./rag_storage'):
        shutil.rmtree('./rag_storage')
        
    # ก๊อปปี้มาวางให้ถูกที่
    shutil.copytree(source_dir, './rag_storage')
    print("✅ จัดเรียงโฟลเดอร์สำเร็จ! ฐานข้อมูลพร้อมใช้งานแล้วครับ")
else:
    print("❌ หาไฟล์ฐานข้อมูลไม่พบ! รบกวนเช็คว่าตอนอัปโหลด .zip สำเร็จเรียบร้อยดีไหมครับ")

✅ พบฐานข้อมูลต้นฉบับซ่อนอยู่ที่: /kaggle/input/datasets/thanananwongjaroen/lag-storage/rag_storage
✅ จัดเรียงโฟลเดอร์สำเร็จ! ฐานข้อมูลพร้อมใช้งานแล้วครับ


In [20]:
import os, time
from pyngrok import ngrok

os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
os.system("fuser -k 8501/tcp")
time.sleep(5)

os.system("nohup streamlit run app.py --server.address 0.0.0.0 --server.port 8501 > streamlit_log.txt 2> streamlit_error.txt &")
time.sleep(20)

ngrok.kill()
ngrok.set_auth_token("3Af1FlQZFRLUWoX2PNcNamqFZc5_d3qxnZHuVWNSbALNqvCp")
public_url = ngrok.connect(8501)
print(f"✅ ลิงก์: {public_url}")

✅ ลิงก์: NgrokTunnel: "https://unfondly-acetimetric-kailey.ngrok-free.dev" -> "http://localhost:8501"


In [8]:
!cat streamlit_error.txt

cat: streamlit_error.txt: No such file or directory


In [9]:
!cat streamlit_log.txt

cat: streamlit_log.txt: No such file or directory


In [10]:
!ls -l

total 12
-rw-r--r-- 1 root root 5744 Mar 15 19:45 app.py
drwxr-xr-x 2 root root 4096 Mar 15 19:42 rag_storage


In [11]:
!ls -la ./rag_storage

total 7252
drwxr-xr-x 2 root root    4096 Mar 15 19:42 .
drwxr-xr-x 4 root root    4096 Mar 15 19:45 ..
-rw-r--r-- 1 root root 4206637 Mar 15 19:42 default__vector_store.json
-rw-r--r-- 1 root root 3145673 Mar 15 19:42 docstore.json
-rw-r--r-- 1 root root      18 Mar 15 19:42 graph_store.json
-rw-r--r-- 1 root root      72 Mar 15 19:42 image__vector_store.json
-rw-r--r-- 1 root root   52541 Mar 15 19:42 index_store.json


In [12]:
!sed -n '88,100p' /kaggle/working/app.py

    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("พิมพ์คำถามที่นี่ เช่น ขอวิธีทำแกงเขียวหวาน"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("กำลังค้นหาสูตรจากฐานข้อมูล..."):
            response = chat_engine.chat(prompt)

            # ตัด <think> ออก (เพิ่ม |$ เผื่อกรณีแท็กปิดไม่สมบูรณ์)
